# AI6130 Assignment 2 - Complete End-to-End Experiment
## Parameter-Efficient Fine-Tuning of Large Language Models

**This notebook performs all required experiments:**
- Fine-tunes 2 models (OpenLLaMA 3B, TinyLlama v1.1)
- Uses 2 PEFT methods (LoRA, AdapterP)
- Evaluates on 2 benchmarks (AQuA, AddSub)
- Generates complete results table for report

**Total experiments: 4 models × 2 benchmarks = 8 evaluations**

In [ ]:
# ============================================================================
# AI6130 ASSIGNMENT 2 - COMPLETE END-TO-END EXPERIMENT PIPELINE
# ============================================================================
# This cell performs all required tasks:
# 1. Environment setup
# 2. Repository cloning
# 3. Fine-tuning 4 model configurations
# 4. Evaluating all models on 2 benchmarks
# 5. Generating comprehensive results table
# ============================================================================

import os
import subprocess
import json
import time
from datetime import datetime
import pandas as pd
import re

# ============================================================================
# SECTION 1: ENVIRONMENT SETUP
# ============================================================================
print("="*80)
print("SECTION 1: ENVIRONMENT SETUP")
print("="*80)

# Disable Weights & Biases tracking (optional)
os.environ["WANDB_MODE"] = "disabled"

# Check if running on Google Colab
try:
    from google.colab import drive
    IS_COLAB = True
    print("✓ Running on Google Colab")
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/AI6130_Assignment2'
    os.makedirs(WORK_DIR, exist_ok=True)
    os.chdir(WORK_DIR)
except:
    IS_COLAB = False
    print("✓ Running on Kaggle or local environment")
    WORK_DIR = '/kaggle/working/AI6130_Assignment2'
    os.makedirs(WORK_DIR, exist_ok=True)
    os.chdir(WORK_DIR)

print(f"✓ Working directory: {WORK_DIR}")

# ============================================================================
# SECTION 2: CLONE REPOSITORY
# ============================================================================
print("\n" + "="*80)
print("SECTION 2: CLONING REPOSITORY")
print("="*80)

REPO_DIR = os.path.join(WORK_DIR, 'AI6130_Assignment2')
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(['git', 'clone', 'https://github.com/duyngtr16061999/AI6130_Assignment2'], check=True)
    print("✓ Repository cloned successfully")
else:
    print("✓ Repository already exists")

os.chdir(REPO_DIR)
print(f"✓ Changed directory to: {os.getcwd()}")

# ============================================================================
# SECTION 3: INSTALL DEPENDENCIES
# ============================================================================
print("\n" + "="*80)
print("SECTION 3: INSTALLING DEPENDENCIES")
print("="*80)

packages = [
    'fire',
    'datasets',
    'transformers==4.38.0',
    'accelerate==0.28.0'
]

for package in packages:
    print(f"Installing {package}...")
    subprocess.run(['pip', 'install', '-q', package], check=True)
    print(f"✓ {package} installed")

# Uninstall conflicting peft if exists
print("Removing conflicting peft package...")
subprocess.run(['pip', 'uninstall', 'peft', '-y'], capture_output=True)
print("✓ Dependencies installed successfully")

# ============================================================================
# SECTION 4: EXPERIMENT CONFIGURATION
# ============================================================================
print("\n" + "="*80)
print("SECTION 4: EXPERIMENT CONFIGURATION")
print("="*80)

# Define all experimental configurations
EXPERIMENTS = [
    {
        'model_name': 'OpenLLaMA 3B',
        'base_model': 'openlm-research/open_llama_3b_v2',
        'adapter': 'lora',
        'adapter_display': 'LoRA',
        'output_dir': './trained_models/openlm-lora'
    },
    {
        'model_name': 'OpenLLaMA 3B',
        'base_model': 'openlm-research/open_llama_3b_v2',
        'adapter': 'AdapterP',
        'adapter_display': 'AdapterP',
        'output_dir': './trained_models/openlm-adapterp'
    },
    {
        'model_name': 'TinyLlama v1.1',
        'base_model': 'TinyLlama/TinyLlama_v1.1',
        'adapter': 'lora',
        'adapter_display': 'LoRA',
        'output_dir': './trained_models/tinyllama-lora'
    },
    {
        'model_name': 'TinyLlama v1.1',
        'base_model': 'TinyLlama/TinyLlama_v1.1',
        'adapter': 'AdapterP',
        'adapter_display': 'AdapterP',
        'output_dir': './trained_models/tinyllama-adapterp'
    }
]

EVALUATION_DATASETS = ['AQuA', 'AddSub']

# Training hyperparameters
TRAINING_CONFIG = {
    'data_path': './ft-training_set/math_7k.json',
    'batch_size': 4,
    'micro_batch_size': 4,
    'num_epochs': 1,
    'learning_rate': 3e-4,
    'cutoff_len': 256,
    'val_set_size': 120
}

print(f"✓ Configured {len(EXPERIMENTS)} model experiments")
print(f"✓ Evaluation on {len(EVALUATION_DATASETS)} benchmarks: {', '.join(EVALUATION_DATASETS)}")
print(f"✓ Total evaluations: {len(EXPERIMENTS)} × {len(EVALUATION_DATASETS)} = {len(EXPERIMENTS) * len(EVALUATION_DATASETS)}")

# ============================================================================
# SECTION 5: FINE-TUNING ALL MODELS
# ============================================================================
print("\n" + "="*80)
print("SECTION 5: FINE-TUNING ALL MODELS")
print("="*80)

training_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*80}")
    print(f"EXPERIMENT {i}/{len(EXPERIMENTS)}: {exp['model_name']} + {exp['adapter_display']}")
    print(f"{'='*80}")
    
    # Check if model already trained
    if os.path.exists(exp['output_dir']):
        print(f"⚠ Model already exists at {exp['output_dir']}")
        print("⚠ Skipping fine-tuning (delete directory to retrain)")
        training_results.append({
            'model': exp['model_name'],
            'adapter': exp['adapter_display'],
            'status': 'Already trained (skipped)',
            'time': 'N/A'
        })
        continue
    
    # Build fine-tuning command
    cmd = [
        'python', 'finetune.py',
        '--base_model', exp['base_model'],
        '--data_path', TRAINING_CONFIG['data_path'],
        '--output_dir', exp['output_dir'],
        '--batch_size', str(TRAINING_CONFIG['batch_size']),
        '--micro_batch_size', str(TRAINING_CONFIG['micro_batch_size']),
        '--num_epochs', str(TRAINING_CONFIG['num_epochs']),
        '--learning_rate', str(TRAINING_CONFIG['learning_rate']),
        '--cutoff_len', str(TRAINING_CONFIG['cutoff_len']),
        '--val_set_size', str(TRAINING_CONFIG['val_set_size']),
        '--adapter_name', exp['adapter']
    ]
    
    print(f"\nCommand: {' '.join(cmd)}\n")
    
    # Execute fine-tuning
    start_time = time.time()
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        elapsed_time = time.time() - start_time
        
        print(f"\n✓ Fine-tuning completed successfully!")
        print(f"✓ Time elapsed: {elapsed_time/60:.2f} minutes")
        print(f"✓ Model saved to: {exp['output_dir']}")
        
        training_results.append({
            'model': exp['model_name'],
            'adapter': exp['adapter_display'],
            'status': 'Success',
            'time': f"{elapsed_time/60:.2f} min"
        })
        
    except subprocess.CalledProcessError as e:
        elapsed_time = time.time() - start_time
        print(f"\n✗ Fine-tuning FAILED!")
        print(f"✗ Error: {e}")
        print(f"\nStderr output:\n{e.stderr}")
        
        training_results.append({
            'model': exp['model_name'],
            'adapter': exp['adapter_display'],
            'status': 'FAILED',
            'time': f"{elapsed_time/60:.2f} min"
        })

# Display training summary
print("\n" + "="*80)
print("FINE-TUNING SUMMARY")
print("="*80)
training_df = pd.DataFrame(training_results)
print(training_df.to_string(index=False))

# ============================================================================
# SECTION 6: EVALUATING ALL MODELS
# ============================================================================
print("\n" + "="*80)
print("SECTION 6: EVALUATING ALL MODELS")
print("="*80)

evaluation_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    # Check if model weights exist
    if not os.path.exists(exp['output_dir']):
        print(f"\n⚠ WARNING: Model weights not found at {exp['output_dir']}")
        print(f"⚠ Skipping evaluation for {exp['model_name']} + {exp['adapter_display']}")
        for dataset in EVALUATION_DATASETS:
            evaluation_results.append({
                'Model': exp['model_name'],
                'PEFT Method': exp['adapter_display'],
                'Dataset': dataset,
                'Accuracy': 'N/A (model not trained)',
                'Status': 'SKIPPED'
            })
        continue
    
    for dataset in EVALUATION_DATASETS:
        print(f"\n{'='*80}")
        print(f"EVALUATION {len(evaluation_results)+1}/{len(EXPERIMENTS)*len(EVALUATION_DATASETS)}")
        print(f"Model: {exp['model_name']} + {exp['adapter_display']}")
        print(f"Dataset: {dataset}")
        print(f"{'='*80}")
        
        # Build evaluation command
        cmd = [
            'python', 'evaluate.py',
            '--adapter', exp['adapter_display'],
            '--dataset', dataset,
            '--base_model', exp['base_model'],
            '--lora_weights', exp['output_dir']
        ]
        
        print(f"\nCommand: {' '.join(cmd)}\n")
        
        # Execute evaluation
        start_time = time.time()
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=1800)
            elapsed_time = time.time() - start_time
            
            # Parse accuracy from output
            output = result.stdout
            print(output[-1000:])  # Print last 1000 chars of output
            
            # Try to extract accuracy using regex
            accuracy_match = re.search(r'Accuracy[:\s]+(\d+\.?\d*)%?', output, re.IGNORECASE)
            if accuracy_match:
                accuracy = accuracy_match.group(1)
                if '.' not in accuracy:
                    accuracy = f"{float(accuracy):.2f}"
            else:
                # Try alternative patterns
                acc_match = re.search(r'acc[uracy]*[:\s]+(\d+\.?\d*)%?', output, re.IGNORECASE)
                if acc_match:
                    accuracy = acc_match.group(1)
                else:
                    accuracy = 'See output above'
            
            print(f"\n✓ Evaluation completed successfully!")
            print(f"✓ Time elapsed: {elapsed_time:.2f} seconds")
            print(f"✓ Accuracy: {accuracy}%")
            
            evaluation_results.append({
                'Model': exp['model_name'],
                'PEFT Method': exp['adapter_display'],
                'Dataset': dataset,
                'Accuracy': f"{accuracy}%",
                'Status': 'Success'
            })
            
        except subprocess.TimeoutExpired:
            elapsed_time = time.time() - start_time
            print(f"\n✗ Evaluation TIMEOUT (exceeded 30 minutes)")
            evaluation_results.append({
                'Model': exp['model_name'],
                'PEFT Method': exp['adapter_display'],
                'Dataset': dataset,
                'Accuracy': 'TIMEOUT',
                'Status': 'TIMEOUT'
            })
            
        except subprocess.CalledProcessError as e:
            elapsed_time = time.time() - start_time
            print(f"\n✗ Evaluation FAILED!")
            print(f"✗ Error: {e}")
            print(f"\nStderr output (last 500 chars):\n{e.stderr[-500:]}")
            
            evaluation_results.append({
                'Model': exp['model_name'],
                'PEFT Method': exp['adapter_display'],
                'Dataset': dataset,
                'Accuracy': 'FAILED',
                'Status': 'FAILED'
            })

# ============================================================================
# SECTION 7: FINAL RESULTS TABLE
# ============================================================================
print("\n" + "="*80)
print("SECTION 7: FINAL RESULTS - READY FOR REPORT")
print("="*80)

# Create results DataFrame
results_df = pd.DataFrame(evaluation_results)

# Pivot table for better visualization
if not results_df.empty:
    pivot_df = results_df.pivot_table(
        index=['Model', 'PEFT Method'],
        columns='Dataset',
        values='Accuracy',
        aggfunc='first'
    )
    
    print("\n📊 RESULTS TABLE (Copy to your report)")
    print("="*80)
    print(pivot_df.to_string())
    
    # Also show flat table
    print("\n\n📋 DETAILED RESULTS TABLE")
    print("="*80)
    display_df = results_df[['Model', 'PEFT Method', 'Dataset', 'Accuracy']]
    print(display_df.to_string(index=False))
    
    # Save to CSV for easy import
    results_csv = 'assignment2_results.csv'
    results_df.to_csv(results_csv, index=False)
    print(f"\n✓ Results saved to: {results_csv}")
    
    # Save pivot table to CSV
    pivot_csv = 'assignment2_results_pivot.csv'
    pivot_df.to_csv(pivot_csv)
    print(f"✓ Pivot table saved to: {pivot_csv}")
    
else:
    print("⚠ No evaluation results available")

# ============================================================================
# SECTION 8: EXPERIMENT SUMMARY
# ============================================================================
print("\n" + "="*80)
print("SECTION 8: EXPERIMENT SUMMARY")
print("="*80)

print(f"\n📈 STATISTICS:")
print(f"  • Total models fine-tuned: {len([r for r in training_results if r['status'] == 'Success'])}")
print(f"  • Total evaluations performed: {len([r for r in evaluation_results if r['Status'] == 'Success'])}")
print(f"  • Failed fine-tuning: {len([r for r in training_results if r['status'] == 'FAILED'])}")
print(f"  • Failed evaluations: {len([r for r in evaluation_results if r['Status'] == 'FAILED'])}")

print(f"\n🎯 ASSIGNMENT REQUIREMENTS COMPLETED:")
print(f"  ✓ Fine-tuned 2 base models (OpenLLaMA 3B, TinyLlama v1.1)")
print(f"  ✓ Used 2 PEFT methods (LoRA, AdapterP)")
print(f"  ✓ Evaluated on 2 benchmarks (AQuA, AddSub)")
print(f"  ✓ Generated results table for report")

print(f"\n📁 OUTPUT FILES:")
print(f"  • Trained models: ./trained_models/")
print(f"  • Results CSV: assignment2_results.csv")
print(f"  • Pivot table CSV: assignment2_results_pivot.csv")

print("\n" + "="*80)
print("✅ ALL EXPERIMENTS COMPLETED!")
print("="*80)
print("\nNext steps for your report:")
print("1. Copy the results table above")
print("2. Analyze why certain methods performed better")
print("3. Discuss parameter efficiency of LoRA vs AdapterP")
print("4. Mention any interesting observations")
print("5. Consider running optional experiments (more epochs, datasets, benchmarks)")
print("\n" + "="*80)